In [ ]:
import os

def generate_image_paths(super_path):
    # Initialize the dictionary
    image_path = {}

    # Traverse the directory structure
    for root, dirs, files in os.walk(super_path):
        for file in files:
            if file.endswith('.png'):
                # Get the relative path from super_path
                relative_path = os.path.relpath(root, super_path)
                # Split the relative path to get <string> and <integer>
                string_part, integer_part = os.path.split(relative_path)
                integer_part = int(integer_part)
                
                # Extract the position from the filename
                file_name_parts = file.split('_')
                position = file_name_parts[-1].split('.')[0]  # Extract 'bottom', 'top', etc.
                
                # Construct the full file path
                full_file_path = os.path.join(root, file)
                
                # Populate the dictionary
                if string_part not in image_path:
                    image_path[string_part] = {}
                if integer_part not in image_path[string_part]:
                    image_path[string_part][integer_part] = {}
                
                image_path[string_part][integer_part][position] = full_file_path

    return image_path



In [ ]:

left_image_path = {}
left_image_path_temp = {}
right_image_path = {}
right_image_path_temp = {}


def find_train_folders(directory):
    train_folders = []
    for root, dirs, files in os.walk(directory):
        for dir in dirs:
            if dir.startswith('train'):
                train_folders.append(os.path.join(root, dir))
    return train_folders


train_path = find_train_folders(
    '/mnt/Velocity Vault/Personal/Projects/Python/Autofocus/Train/')


for path in train_path:
    left_image_path_temp = generate_image_paths(path+'/raw_up_left_pd')
    left_image_path.update(left_image_path_temp)
    right_image_path_temp = generate_image_paths(path+'/raw_up_right_pd')
    right_image_path.update(right_image_path_temp)


import pprint

pprint.pprint(left_image_path)
pprint.pprint(right_image_path)
print(len(left_image_path))
print(len(right_image_path))


In [ ]:
def images_path_to_dataset_path(image_path,count=0,dataset=[]):
    for image_type in image_path:
        if count==len(dataset):
            dataset.append([])
            dataset.append([])
            dataset.append([])
            dataset.append([])
            dataset.append([])
        for focal_slice in image_path[image_type]:
            dataset[count].append(image_path[image_type][focal_slice]['top'])
            dataset[count+1].append(image_path[image_type][focal_slice]['bottom'])
            dataset[count+2].append(image_path[image_type][focal_slice]['center'])
            dataset[count+3].append(image_path[image_type][focal_slice]['left'])
            dataset[count+4].append(image_path[image_type][focal_slice]['right'])
        count+=5
    return dataset


dataset_path=images_path_to_dataset_path(left_image_path)
dataset_path=images_path_to_dataset_path(right_image_path,dataset=dataset_path)
pprint.pprint(dataset_path)
print(len(dataset_path))

In [ ]:
import copy
left_images=copy.deepcopy(left_image_path)
right_images=copy.deepcopy(right_image_path)

def load_images(image_path):

    images=copy.deepcopy(image_path)

    for image_type in image_path:
        for focal_slice in image_path[image_type]:
            for pos in image_path[image_type][focal_slice]:
                images[image_type][focal_slice][pos]=